In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
hugging_face_token = os.getenv("HF_TOKEN")
groq_api_key = os.getenv("GROQ_API_KEY")

In [4]:
model = ChatGroq(model = 'Llama3-8b-8192', groq_api_key = groq_api_key)

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [5]:
from langchain_core.documents import Document

In [6]:
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [7]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

### Vector Embedding

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
#sentence transformer is an embedding that map sentences to vector
embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2') #sentence-transformers/all-MiniLM-L6-v2 sentence-transformers/all-MiniLM-L6-v2

2026-01-03 10:35:43.574656: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


### Vector store

In [2]:
from langchain_chroma import Chroma

In [ ]:
#This maps the document content with the embedding values (numbers) and store it in Chroma
vector_score = Chroma.from_documents(documents = documents, embedding = embeddings)

vector_score

In [ ]:
vector_score.similarity_search('cat')

[Document(id='149e3462-276e-4241-8f01-9167a03d1172', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='9fe0d6ee-7407-465c-9db3-33bc09adf18c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='9257137b-b8b6-426e-a1a3-0e82d899196e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='334813ff-2ba4-4ab0-a4df-5ce4ef244323', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [ ]:
vector_score.similarity_search_with_score('cat')

[(Document(id='149e3462-276e-4241-8f01-9167a03d1172', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='9fe0d6ee-7407-465c-9db3-33bc09adf18c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740901231765747),
 (Document(id='9257137b-b8b6-426e-a1a3-0e82d899196e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956906080245972),
 (Document(id='334813ff-2ba4-4ab0-a4df-5ce4ef244323', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.66579270362854)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [ ]:
from typing import List

from langchain_core.runnables import RunnableLambda

##### first method to retrieve content from store

In [ ]:
#now retrieve data from the vector store by returning the top result 
retriever = RunnableLambda(vector_score.similarity_search).bind(k=1)
#retrieve item in list from the vector store
retriever.batch(['cat', 'dog'])

[[Document(id='149e3462-276e-4241-8f01-9167a03d1172', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9fe0d6ee-7407-465c-9db3-33bc09adf18c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

#### second method to retrieve content from vector store - This is the best to query from vector store

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [ ]:
retriever = vector_score.as_retriever(
    search_type = 'similarity',
    search_kwargs={'k': 1} # top result will be returned
)

In [ ]:
retriever.batch(['cat', 'dog'])

[[Document(id='149e3462-276e-4241-8f01-9167a03d1172', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9fe0d6ee-7407-465c-9db3-33bc09adf18c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

### Implement basic Retrieval Augmented generation (RAG) using the retrieval above

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
#define your prompt template, with two input variables
prompt_template = """
Answer the given {question} using the {context} provided only, say you don't know if a question is outside this {context}

question : {question}
context : {context}

"""

In [ ]:
#since you're defining a single prompt (not in chat form that requires system, human or ai) use the "from_template"
prompt = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
#define a way to extract the string content from the document page content as a single string
def doc_format(docs):
    return "\n".join(doc.page_content for doc in docs)

In [ ]:
#now, define the key-value pair input variable needeed in your prompt then chain it with prompt template then to the model before passing it to sttring output
rag_chain = (
    {
        'context': retriever | doc_format,
        'question': RunnablePassthrough()
    }
    | prompt
    | model
    |StrOutputParser()
)

In [ ]:
#ask question from using this chain
response = rag_chain.invoke('tell me about cat')

print(response)

Cats are independent pets that often enjoy their own space.


In [ ]:
#ask question from using this chain
response = rag_chain.invoke('tell me about ridwan')

print(response)

I'm not sure what you mean by "Ridwan". Could you please provide more context or clarify who or what Ridwan is? Based on the given context, I don't have any information about a person or entity named Ridwan.


In [ ]:
#ask question from using this chain
response = rag_chain.invoke('tell me about dog')

print(response)

According to the context, dogs are great companions, known for their loyalty and friendliness.
